[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/projective_loop/projective_loop.ipynb)

# Loop Subdivision of Weighted Points

This example computes Loop subdivision of a triangle mesh whose vertices are weighted points. Loop subdivision smooths a mesh: every triangle splits in four, every vertex moves to an average of itself and its neighbours, every edge gets a new vertex between its ends, and repeated steps converge to a smooth surface. Here the vertices are points of projective space, each with its own weight, and the averages are taken of the points as they are, weights included. Two things follow. A vertex's weight is a handle on the surface: a heavy vertex pulls the surface towards itself, a light one lets it go. And subdivision commutes with perspective: refining and then viewing in perspective gives exactly what viewing in perspective and then refining gives, where averaging positions does not.

The points are points of projective geometric algebra, and one Loop step is a sparse extensor with scalar cells, a map from the coarser mesh's vertex sites to the finer one's, `step * mesh.vertices`, and the steps compose into one map. A perspective is a map on points, the identity plus one term, `Point + centre * (axis & Point) * strength`. The step acts across the sites and the perspective on the point at each site, so the two commute.

In the homogeneous-coordinate notation of rational curves and surfaces, each vertex reads as $(w\,\mathbf p,\ w)$, and this is rational Loop subdivision, its weights playing the part they play in rational Bézier patches and NURBS.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from numga.sparse import SparseExtensor
from examples import instantiate
from examples.animation import save_figure, show_animation
from examples.surfaces.projective_loop import render

mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()
Point = ga.gatype.antivector()
PointMap = ga.gatype((Point, Point))                                           # Point <- Point
Mesh = instantiate("examples.mesh", ga).Mesh


# The octahedron: its vertices, the one at `top` on top, and its faces, counter-clockwise from outside.
octahedron = np.array([[1.0, 0.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, -1.0, 0.0], [0.0, 0.0, 1.0], [0.0, 0.0, -1.0]])
top = 4
octahedron_faces = np.array([[0, 2, 4], [2, 1, 4], [1, 3, 4], [3, 0, 4], [2, 0, 5], [1, 2, 5], [3, 1, 5], [0, 3, 5]])

## 1. A Loop step as a map on the vertex sites

An old vertex of valence $n$ keeps $1 - n\beta$ of itself and takes $\beta$ of each neighbour, with $\beta = \tfrac1n\big(\tfrac58 - (\tfrac38 + \tfrac14\cos\tfrac{2\pi}{n})^2\big)$; a new vertex on an edge takes $\tfrac38$ of each end and $\tfrac18$ of each vertex across the edge. These shares depend only on how the mesh is connected, so each step is built once from the faces, as `Mesh.loop`: a sparse map from the vertex sites of the coarse mesh to those of the fine one, the old vertices followed by one per edge, its cells the shares. It acts on whatever sits at the sites, here points with their weights, and the three steps compose into one map, from the octahedron's six vertex sites to the 258 of the refined mesh.

In [ ]:
# Loop steps taken.
n_loop_steps = 3

In [ ]:
def refinements(mesh: Mesh, pinned: np.ndarray, n_steps: int) -> Iterator[tuple[SparseExtensor, Mesh]]:
    """The map of each of so many Loop steps from a mesh, with the finer mesh it gives."""
    for _ in range(n_steps):
        step = mesh.loop(pinned)                                               # [finer, coarser] Scalar
        mesh, pinned = Mesh(step * mesh.vertices, mesh.split_faces), mesh.split_pins(pinned)
        yield step, mesh


def subdivision(mesh: Mesh, n_steps: int) -> tuple[SparseExtensor, Mesh]:
    """So many Loop steps from a closed mesh, nothing pinned, composed into one map from its vertex sites to
    those of the refined mesh, and the refined mesh."""
    steps, meshes = zip(*refinements(mesh, np.zeros(mesh.vertices.batch().shape[-1], dtype=bool), n_steps))
    return reduce(lambda total, step: step * total, steps), meshes[-1]       # [refined, vertices] Scalar


subdivide, refined = subdivision(Mesh(mv.point(octahedron).field(), octahedron_faces), n_loop_steps)
print(subdivide.gatype, subdivide.shape)

## 2. Weights pull and release

The same steps refine the octahedron with all weights alike, with its top vertex four times as heavy, and with it four times as light, all three as one batch. With equal weights the surface is ordinary Loop subdivision of the positions. The heavy top vertex draws the surface up towards it; the light one lets it sink back towards the middle. The dotted lines are the control mesh. The animation sweeps the top weight from a quarter to four and back, each frame's octahedron through the one composed map.

In [ ]:
# The weights of the vertices per case: all alike, the top one heavy, the top one light.
weights = np.array([[1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 4.0, 1.0], [1.0, 1.0, 1.0, 1.0, 0.25, 1.0]])
# The top vertex's weight swept from light to heavy and back, the frames of the sweep, and the duration
# of each.
sweep = (0.25, 4.0)
n_sweep_frames = 48
duration_ms = 60

In [ ]:
controls = (mv.point(octahedron) * weights).field()                            # [cases] Point[vertices]
surfaces = subdivide * controls                                                # [cases] Point[refined]

save_figure(render.draw_weights(controls, octahedron_faces, surfaces, refined.faces), "projective_loop_weights");

In [ ]:
phase = np.linspace(0, 2 * np.pi, n_sweep_frames, endpoint=False)
top_weight = np.exp(np.log(sweep).mean() - np.log(sweep[1] / sweep[0]) / 2 * np.cos(phase))   # [frames]
sweep_weights = np.where(np.arange(len(octahedron)) == top, top_weight[:, None], 1.0)   # [frames, vertices]
swept = ((mv.point(octahedron) * weight).field() for weight in sweep_weights)  # Point[vertices] each

show_animation(render.animate_weights(((control, subdivide * control) for control in swept), octahedron_faces, refined.faces),
               "projective_loop_sweep", duration_ms)

## 3. Subdivision commutes with perspective

A perspective here is a central collineation: each point moves along its line through a centre, by the centre times its pairing with an axis plane, which leaves the axis in place and changes the weight of every point off it. It is the identity plus one more term, `Point + centre * (axis & Point) * strength`. The Loop step acts across the vertex sites and the collineation on the point at each site, so the two commute exactly: refining and then viewing equals viewing and then refining. Averaging positions instead divides each point by its weight first, and the collineation changes the weights, so there the two orders part. Each surface is coloured by how far the other order's surface lies, on one scale for both panels: with weights the two agree everywhere, with positions they part most where the perspective changes the weights most.

In matrix notation, with the homogeneous vertex coordinates the rows of a matrix $V$, a Loop step is a sparse matrix $S$ acting from the left and the perspective a $4 \times 4$ matrix $H$ acting from the right, so that $S\,(V H^\top) = (S V)\,H^\top$; dividing each row by its last entry first puts a diagonal matrix between them that depends on $H$.

In [ ]:
# The octahedron stretched and set in front of the perspective, and the perspective: its centre at the
# origin, its axis the plane z = 0, and its strength.
stretch = np.array([1.0, 1.0, 1.5])
offset = np.array([0.0, 0.0, 1.6])
strength = 1.0

In [ ]:
def collineation(centre: Point, axis: Plane, strength: float) -> PointMap:
    """The central collineation of a centre and an axis plane: each point moves along its line through the
    centre, by the centre times its pairing with the axis; points on the axis stay where they are."""
    return Point + centre * (axis & Point) * strength                         # [] Point <- Point


def unit(points: Point) -> Point:
    """The points divided by their weights: their positions, at unit weight."""
    return points / (mv.w & points)


shape = mv.point(octahedron * stretch + offset).field()                        # Point[vertices]
seen = collineation(mv.w.dual(), mv.z, strength)                               # [] Point <- Point
weighted_orders = stack([subdivide * seen(shape), seen(subdivide * shape)])     # [orders] Point[refined]
positional_orders = stack([subdivide * unit(seen(shape)), seen(subdivide * unit(shape))])   # [orders] Point[refined]

print(f"largest gap between the orders, with weights: {render.separation(weighted_orders):.1e}")
print(f"largest gap between the orders, positions:    {render.separation(positional_orders):.3f}")
save_figure(render.draw_orders(seen(shape), octahedron_faces, weighted_orders, positional_orders, refined.faces), "projective_loop_perspective");

In [ ]:
# checks
assert render.separation(weighted_orders) < 1e-12
assert render.separation(positional_orders) > 1e-2
# With all weights alike the two schemes agree.
assert render.separation(stack([surfaces[0], unit(surfaces[0])])) < 1e-12